In [1]:
!pip install transformers datasets evaluate jiwer peft accelerate librosa

In [2]:
!pip install -U torchao
!pip install transformers datasets evaluate jiwer peft accelerate librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
# from datasets import load_dataset

# # Load the combined test dataset
# dataset = load_dataset("abhi8799/Punjabi-STT_Combined", split="test")

In [4]:
import evaluate

wer_metric = evaluate.load("wer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:


from transformers import pipeline
import torch
import librosa
import os

# 1. Paths to your downloaded Zoho WorkDrive files inside Colab
audio_file_path = "/content/Full Narration_MarauliKhurad.m4a"
transcript_file_path = "/content/full narration_maraulikhurad_gurumukhi.txt"

# 2. Read the ground truth text from your uploaded text file
with open(transcript_file_path, "r", encoding="utf-8") as f:
    ground_truth_text = f.read().strip()

# Load and resample audio to Whisper's required 16kHz
audio_array, sampling_rate = librosa.load(audio_file_path, sr=16000)

models_to_test = [
     "Garden2006/whisper-large-v3-turbo-gurmukhi-lora",
     "abhi8799/whisper-large-v3-turbo-gurmukhi-lora",
     "abhi8799/whisper-small-gurmukhi-lora",
     "KaliNangia/whisper-large-v3-turbo-gurmukhi-lora",
     "KaliNangia/whisper-medium-gurmukhi-lora"
]

results = {}

# Run inference across remaining models
for model_id in models_to_test:
    print(f"Evaluating {model_id}...")
    try:
        # Pipeline execution
        asr_pipeline = pipeline(
            "automatic-speech-recognition",
            model=model_id,
            device=0 if torch.cuda.is_available() else -1,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )

        # Generate transcription
        prediction = asr_pipeline(
            {"array": audio_array, "sampling_rate": sampling_rate},
            chunk_length_s=30,
            return_timestamps=True,
            generate_kwargs={
                "language": "punjabi",
                "task": "transcribe"
            }
        )["text"]

        # --- NEW FORMATTING CODE FOR THE COMMIT REQS ---
        # Extracts 'abhi8799' from 'abhi8799/whisper-small-gurmukhi-lora'
        hf_username = model_id.split("/")[0]

        # Maps the model string to the simple clean name format requested
        if "turbo" in model_id:
            model_short = "turbo"
        elif "medium" in model_id:
            model_short = "medium"
        elif "small" in model_id:
            model_short = "small"
        else:
            model_short = "large"

        transcript_output_path = f"{hf_username}_{model_short}.txt"

        with open(transcript_output_path, "w", encoding="utf-8") as out_f:
            out_f.write(prediction)
        print(f"💾 Saved transcript following rule to: {transcript_output_path}")
        # -------------------------------------------------------------

        # Calculate WER score
        wer_score = wer_metric.compute(predictions=[prediction], references=[ground_truth_text])
        results[model_id] = wer_score

        print(f"WER for {model_id}: {wer_score:.4f}\n")

    except Exception as e:
        print(f"❌ Error running {model_id}: {e}\n")

/tmp/ipykernel_1629/501668352.py:125: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sampling_rate = librosa.load(audio_file_path, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Evaluating Garden2006/whisper-large-v3-turbo-gurmukhi-lora...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


...

💾 Saved transcript following rule to: KaliNangia_turbo.txt
WER for KaliNangia/whisper-large-v3-turbo-gurmukhi-lora: 0.8020

Evaluating KaliNangia/whisper-medium-gurmukhi-lora...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


💾 Saved transcript following rule to: KaliNangia_medium.txt
WER for KaliNangia/whisper-medium-gurmukhi-lora: 0.8272



In [11]:
import pandas as pd

# 1. Convert your results dictionary into a structured table
report_data = []
for model_name, wer_score in results.items():
    report_data.append({
        "Model Name": model_name,
        "Word Error Rate (WER)": f"{wer_score:.4f}",
        "Status": "Completed"
    })

df = pd.DataFrame(report_data)

# 2. Save it as a Markdown file (.md) or CSV (.csv)
report_filename = "WER_Finetuned_Models_Report.md"

with open(report_filename, "w") as f:
    f.write("# Team Fine-Tuned Models Evaluation Report\n\n")
    f.write(f"**Evaluation Dataset:** `abhi8799/Punjabi-STT_Combined` (Test Split)\n\n")
    f.write(df.to_markdown(index=False))

print(f"✅ Report successfully generated and saved as {report_filename}!")

✅ Report successfully generated and saved as WER_Finetuned_Models_Report.md!
